In [0]:
%run ../_shared/00_project_config

In [0]:
%run ../_shared/01_spark_utils

In [0]:
from pyspark.sql import functions as F

In [0]:
# -----------------------------------------------------------------------------
# Configuration
# -----------------------------------------------------------------------------

SOURCE_SYSTEM = "football-data.org"

RAW_MATCHES_ROOT = (
    f"{FOOTBALL_DATA_ORG_RAW_ROOT}/matches"
)

BRONZE_TABLE = (
    f"{BRONZE_SCHEMA}."
    "football_data_org_matches"
)


RAW_MATCH_FILE_NAMES = {
    "PL": "pl_premier_league.json",
    "PD": "pd_la_liga.json",
    "BL1": "bl1_bundesliga.json",
    "SA": "sa_serie_a.json",
    "FL1": "fl1_ligue_1.json",
}

# -----------------------------------------------------------------------------
# Load football-data.org match JSON files into Bronze
# -----------------------------------------------------------------------------
#
# Each raw JSON file contains a response envelope with:
#
#   filters
#   resultSet
#   competition
#   matches[]
#
# Bronze removes only the API response envelope by exploding matches[].
# Individual match fields and nested structures remain source-oriented.
#
# Resulting Bronze grain:
#
#   one row = one football-data.org match
#
# Technical metadata:
#
#   _league
#   _ingested_at
#   _source_file
#   _source_system
# -----------------------------------------------------------------------------

loaded_file_count = 0
loaded_match_count = 0


for competition_code, league in FOOTBALL_DATA_ORG_COMPETITIONS.items():

    file_name = RAW_MATCH_FILE_NAMES[
        competition_code
    ]

    file_path = (
        f"{RAW_MATCHES_ROOT}/"
        f"{file_name}"
    )
    # Track the exact content of the raw API response.
    source_hash = calculate_file_sha256(file_path)

    # -------------------------------------------------------------------------
    # Read raw JSON
    # -------------------------------------------------------------------------
    # Raw files are pretty-printed JSON documents, so multiLine must be enabled.
    # -------------------------------------------------------------------------

    source_df = (
        spark.read
        .option("multiLine", "true")
        .json(file_path)
    )

    # -------------------------------------------------------------------------
    # Explode source matches array
    # -------------------------------------------------------------------------
    #
    # We unpack only the transport-level response envelope.
    # Match field names and nested source structures are preserved.
    # -------------------------------------------------------------------------

    bronze_df = (
        source_df
        .select(
            F.explode("matches").alias("match"),
            F.col("_metadata.file_path").alias("_source_file"),
        )
        .select(
            "match.*",
            "_source_file",
        )
        .withColumn(
            "_season_id",
            F.col("season.id").cast("long"),
        )
        .withColumn(
            "_league",
            F.lit(league),
        )
        .withColumn(
            "_ingested_at",
            F.current_timestamp(),
        )
        .withColumn(
            "_source_system",
            F.lit(SOURCE_SYSTEM),
        )
    )

    # -------------------------------------------------------------------------
    # Incremental loading: identify league-season snapshot
    # -------------------------------------------------------------------------

    season_ids = [
        row["_season_id"]
        for row in (
            bronze_df
            .select("_season_id")
            .distinct()
            .collect()
        )
    ]

    if len(season_ids) != 1 or season_ids[0] is None:
        raise ValueError(
            f"Expected exactly one valid season in {file_path}. "
            f"Found: {season_ids}"
        )

    season_id = season_ids[0]

    replace_predicate = (
        f"_league = '{league}' "
        f"AND _season_id = {season_id}"
    )

    if not bronze_file_needs_processing(
        spark=spark,
        table_name=BRONZE_TABLE,
        replace_predicate=replace_predicate,
        source_hash=source_hash,
    ):
        print(
            f"SKIP unchanged API matches: {file_path}"
        )
        continue

    match_count = bronze_df.count()

    # -------------------------------------------------------------------------
    # Write to Bronze Delta table
    # -------------------------------------------------------------------------

    (

    write_bronze_file_snapshot(
        df=bronze_df,
        table_name=BRONZE_TABLE,
        replace_predicate=replace_predicate,
        source_hash=source_hash,
    )

    )

    loaded_file_count += 1
    loaded_match_count += match_count

    print(
        f"Loaded football-data.org Bronze file "
        f"{loaded_file_count}: "
        f"{competition_code} / {league} | "
        f"matches={match_count} "
        f"-> {BRONZE_TABLE}"
    )

# -----------------------------------------------------------------------------
# Completion message
# -----------------------------------------------------------------------------

print(
    f"Successfully loaded {loaded_file_count} "
    "football-data.org match source files into Bronze. "
    f"Total matches loaded: {loaded_match_count}."
)